# Deep Learning Lab Practical 6
**SVNIT Surat | GPU: P100 | Python 3.10**

Topics covered: VAE, Vanilla GAN, CGAN, DCGAN, GAN Loss Variants, Augmentation Study

---

In [ ]:
# ─────────────────────────────────────────
# CELL 0 — Environment Setup (ALWAYS RUN FIRST)
# ─────────────────────────────────────────
!pip install -q torchmetrics scikit-image umap-learn tqdm

import os, random, numpy as np, torch
os.makedirs("/kaggle/working/outputs", exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

---
## Shared Utilities
Helper functions used across all problems: image grid saving, FID computation, loss plotting, and seed setting.

In [ ]:
# ─────────────────────────────────────────
# CELL 1 — Shared Utility Functions
# ─────────────────────────────────────────
import torch, os, random
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import torchvision.utils as vutils
from torchmetrics.image.fid import FrechetInceptionDistance


def set_seed(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_image_grid(images: torch.Tensor, path: str, nrow: int = 8):
    """
    Save a grid of images as a PNG.
    Accepts tensors in [-1, 1] or [0, 1] range; auto-normalises to [0, 1].
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    images = images.detach().cpu()
    if images.min() < 0:          # [-1,1] → [0,1]
        images = (images + 1) / 2
    images = images.clamp(0, 1)
    grid = vutils.make_grid(images, nrow=nrow, padding=2, normalize=False)
    fig, ax = plt.subplots(figsize=(nrow * 1.5, max(2, images.shape[0] // nrow * 1.5)))
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray' if grid.shape[0] == 1 else None)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [✓] Saved grid → {path}")


def compute_fid(real_loader, fake_images: torch.Tensor, device, max_real: int = 2048) -> float:
    """
    Compute FID score using torchmetrics.image.FrechetInceptionDistance.
    fake_images: tensor of shape (N, C, H, W) in [-1,1] or [0,1].
    Inception expects (N, 3, 299, 299) uint8; we resize and repeat channels.
    """
    import torch.nn.functional as F
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)

    # --- real images ---
    real_count = 0
    for imgs, _ in real_loader:
        if real_count >= max_real:
            break
        imgs = imgs.to(device)
        if imgs.min() < 0:
            imgs = (imgs + 1) / 2
        imgs = imgs.clamp(0, 1)
        if imgs.shape[1] == 1:          # grayscale → 3-channel
            imgs = imgs.repeat(1, 3, 1, 1)
        imgs = F.interpolate(imgs, size=(299, 299), mode='bilinear', align_corners=False)
        fid_metric.update(imgs.float(), real=True)
        real_count += imgs.shape[0]

    # --- fake images ---
    fake = fake_images.detach().cpu()
    if fake.min() < 0:
        fake = (fake + 1) / 2
    fake = fake.clamp(0, 1)
    if fake.shape[1] == 1:
        fake = fake.repeat(1, 3, 1, 1)
    bs = 64
    for i in range(0, fake.shape[0], bs):
        batch = fake[i:i+bs].to(device)
        batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
        fid_metric.update(batch.float(), real=False)

    fid_score = fid_metric.compute().item()
    fid_metric.reset()
    del fid_metric
    torch.cuda.empty_cache()
    return fid_score


def plot_losses(g_losses: list, d_losses: list, save_path: str, title: str = "Training Losses"):
    """Plot G and D loss curves on the same axes and save as PNG."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(g_losses, label='Generator Loss', color='#E74C3C', linewidth=2)
    ax.plot(d_losses, label='Discriminator Loss', color='#3498DB', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=13)
    ax.set_ylabel('Loss', fontsize=13)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [✓] Saved loss curve → {save_path}")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)
print("Utilities loaded. Device:", DEVICE)

---
## Problem 1 — Variational Autoencoder (VAE)

**Architecture:**
- **Encoder:** 784 → FC(512) → FC(256) → μ (FC→latent_dim) + logσ² (FC→latent_dim)
- **Reparameterisation:** z = μ + ε·exp(0.5·logσ²), ε ~ N(0,I)
- **Decoder:** FC(256) → FC(512) → FC(784) → Sigmoid
- **Loss:** BCE_recon + β·KL, with KL annealing (β ramps 0→1 over first 10 epochs)

**Tasks:** Train on FashionMNIST & EMNIST, visualise reconstructions, latent traversals, random samples, latent dim study, t-SNE.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PROBLEM 1 — Variational Autoencoder (VAE)
# ═══════════════════════════════════════════════════════════════

# ── CONFIG ──────────────────────────────────────────────────────
CONFIG = {
    "latent_dims"  : [8, 32, 128],   # dims to benchmark
    "primary_dim"  : 32,             # used for visualisation tasks
    "epochs"       : 30,
    "batch_size"   : 128,
    "lr"           : 1e-3,
    "kl_anneal"    : 10,             # ramp β over first N epochs
    "tsne_samples" : 1000,
    "out_dir"      : "/kaggle/working/outputs/vae",
    "data_dir"     : "/kaggle/working/data",
    "num_workers"  : 2,
}

# ── Imports ─────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, os, time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm
from sklearn.manifold import TSNE
import pandas as pd

set_seed(42)
os.makedirs(CONFIG["out_dir"], exist_ok=True)


# ── Dataset Loaders ─────────────────────────────────────────────
def get_loaders(dataset_name: str, batch_size: int = 128):
    transform = transforms.Compose([transforms.ToTensor()])
    if dataset_name == "fmnist":
        train_ds = datasets.FashionMNIST(CONFIG["data_dir"], train=True,  download=True, transform=transform)
        test_ds  = datasets.FashionMNIST(CONFIG["data_dir"], train=False, download=True, transform=transform)
    elif dataset_name == "emnist":
        train_ds = datasets.EMNIST(CONFIG["data_dir"], split="balanced", train=True,  download=True, transform=transform)
        test_ds  = datasets.EMNIST(CONFIG["data_dir"], split="balanced", train=False, download=True, transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=CONFIG["num_workers"], pin_memory=True)
    return train_loader, test_loader, train_ds, test_ds


# ── Model Definition ─────────────────────────────────────────────
class VAE(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.enc_fc1  = nn.Linear(784, 512)
        self.enc_fc2  = nn.Linear(512, 256)
        self.enc_mu   = nn.Linear(256, latent_dim)
        self.enc_logv = nn.Linear(256, latent_dim)   # logσ²

        # Decoder
        self.dec_fc1  = nn.Linear(latent_dim, 256)
        self.dec_fc2  = nn.Linear(256, 512)
        self.dec_fc3  = nn.Linear(512, 784)

    def encode(self, x):
        x = x.view(-1, 784)
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.enc_mu(h), self.enc_logv(h)

    def reparameterise(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu   # deterministic at eval

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_fc3(h))   # [0,1]

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


# ── Loss ─────────────────────────────────────────────────────────
def vae_loss(recon_x, x, mu, logvar, beta: float = 1.0):
    """
    ELBO loss:
      BCE_recon measures pixel-wise reconstruction.
      KL = -0.5 * sum(1 + logσ² - μ² - σ²)
    """
    x_flat = x.view(-1, 784)
    bce = F.binary_cross_entropy(recon_x, x_flat, reduction='sum') / x.shape[0]
    kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]
    return bce + beta * kl, bce.item(), kl.item()


# ── Training Loop ────────────────────────────────────────────────
def train_vae(model, train_loader, optimizer, epoch: int, kl_anneal_epochs: int):
    model.train()
    beta = min(1.0, epoch / kl_anneal_epochs)   # KL annealing
    total_loss, total_recon, total_kl = 0., 0., 0.
    for imgs, _ in train_loader:
        imgs = imgs.to(DEVICE)
        optimizer.zero_grad()
        recon, mu, logvar = model(imgs)
        loss, recon_l, kl_l = vae_loss(recon, imgs, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        total_loss  += loss.item()
        total_recon += recon_l
        total_kl    += kl_l
    n = len(train_loader)
    return total_loss / n, total_recon / n, total_kl / n


@torch.no_grad()
def eval_vae(model, test_loader):
    model.eval()
    total_mse = 0.
    for imgs, _ in test_loader:
        imgs = imgs.to(DEVICE)
        recon, _, _ = model(imgs)
        total_mse += F.mse_loss(recon, imgs.view(-1, 784), reduction='sum').item()
    return total_mse / (len(test_loader.dataset) * 784)


# ── Visualisation Helpers ────────────────────────────────────────
@torch.no_grad()
def visualise_reconstructions(model, test_loader, save_path, n_pairs=8):
    """Task 2: 8 real vs reconstructed image pairs side-by-side."""
    model.eval()
    imgs, _ = next(iter(test_loader))
    imgs = imgs[:n_pairs].to(DEVICE)
    recon, _, _ = model(imgs)
    recon = recon.view(-1, 1, 28, 28)

    comparison = torch.cat([imgs.cpu(), recon.cpu()], dim=0)  # real then recon
    # Interleave: real[0], recon[0], real[1], recon[1], ...
    interleaved = torch.stack(
        [t for pair in zip(imgs.cpu(), recon.cpu()) for t in pair]
    )
    save_image_grid(interleaved, save_path, nrow=n_pairs)


@torch.no_grad()
def latent_traversal(model, test_loader, save_path, n_steps=10):
    """Task 3: encode 2 images, linearly interpolate, decode."""
    model.eval()
    imgs, _ = next(iter(test_loader))
    imgs = imgs[:2].to(DEVICE)
    mu1, _ = model.encode(imgs[0:1])
    mu2, _ = model.encode(imgs[1:2])
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE)
    z_interp = torch.stack([(1 - a) * mu1 + a * mu2 for a in alphas]).squeeze(1)
    decoded   = model.decode(z_interp).view(-1, 1, 28, 28)
    # Prepend original images
    strip = torch.cat([imgs[0:1].cpu(), decoded.cpu(), imgs[1:2].cpu()])
    save_image_grid(strip, save_path, nrow=n_steps + 2)


@torch.no_grad()
def random_sampling(model, save_path, n=25, latent_dim=32):
    """Task 4: sample z ~ N(0,I), decode, display 5×5 grid."""
    model.eval()
    z = torch.randn(n, latent_dim, device=DEVICE)
    samples = model.decode(z).view(-1, 1, 28, 28)
    save_image_grid(samples, save_path, nrow=5)


@torch.no_grad()
def tsne_latent(model, test_loader, save_path, n_samples=1000):
    """Task 6: t-SNE of latent vectors coloured by class."""
    model.eval()
    all_mu, all_labels = [], []
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        mu, _ = model.encode(imgs)
        all_mu.append(mu.cpu().numpy())
        all_labels.append(labels.numpy())
        if sum(a.shape[0] for a in all_mu) >= n_samples:
            break
    mu_np  = np.concatenate(all_mu,    axis=0)[:n_samples]
    lab_np = np.concatenate(all_labels, axis=0)[:n_samples]

    print("  Running t-SNE …")
    perp = min(30, n_samples - 1)
    tsne_emb = TSNE(n_components=2, perplexity=perp, random_state=42, n_jobs=-1).fit_transform(mu_np)

    n_classes = int(lab_np.max()) + 1
    cmap = plt.get_cmap('tab20', n_classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    for c in range(n_classes):
        mask = lab_np == c
        ax.scatter(tsne_emb[mask, 0], tsne_emb[mask, 1],
                   c=[cmap(c)], label=str(c), s=15, alpha=0.8)
    ax.legend(title='Class', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.set_title('t-SNE of VAE Latent Space', fontsize=14, fontweight='bold')
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [✓] Saved t-SNE → {save_path}")


# ── Full Training Run ────────────────────────────────────────────
def run_vae_experiment(latent_dim: int, dataset_name: str, epochs: int,
                       do_visualise: bool = True):
    tag = f"{dataset_name}_z{latent_dim}"
    print(f"\n{'='*60}")
    print(f"  VAE | dataset={dataset_name.upper()} | latent_dim={latent_dim}")
    print(f"{'='*60}")

    train_loader, test_loader, _, _ = get_loaders(dataset_name, CONFIG["batch_size"])

    model     = VAE(latent_dim=latent_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    history = {"loss": [], "recon": [], "kl": []}
    start_t = time.time()

    for epoch in tqdm(range(1, epochs + 1), desc=f"VAE [{tag}]"):
        l, r, k = train_vae(model, train_loader, optimizer, epoch, CONFIG["kl_anneal"])
        history["loss"].append(l)
        history["recon"].append(r)
        history["kl"].append(k)

    train_time_min = (time.time() - start_t) / 60
    recon_mse = eval_vae(model, test_loader)
    print(f"  Recon MSE = {recon_mse:.6f} | Train time = {train_time_min:.1f} min")

    # ── Loss Curves ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history["recon"], color='#E74C3C', linewidth=2, label='Recon (BCE)')
    axes[0].set_title('Reconstruction Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[1].plot(history["kl"],   color='#3498DB', linewidth=2, label='KL Divergence')
    axes[1].set_title('KL Divergence per Epoch', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('KL'); axes[1].grid(alpha=0.3)
    axes[1].legend()
    fig.suptitle(f'VAE Loss — {dataset_name.upper()} | z={latent_dim}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    loss_path = os.path.join(CONFIG["out_dir"], f"loss_curves_{tag}.png")
    plt.savefig(loss_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [✓] Saved loss curves → {loss_path}")

    if do_visualise:
        # Task 2 — Reconstructions
        visualise_reconstructions(
            model, test_loader,
            os.path.join(CONFIG["out_dir"], f"reconstructions_{tag}.png")
        )
        # Task 3 — Latent traversal
        latent_traversal(
            model, test_loader,
            os.path.join(CONFIG["out_dir"], f"traversal_{tag}.png")
        )
        # Task 4 — Random sampling
        random_sampling(
            model,
            os.path.join(CONFIG["out_dir"], f"random_samples_{tag}.png"),
            n=25, latent_dim=latent_dim
        )
        # Task 6 — t-SNE
        tsne_latent(
            model, test_loader,
            os.path.join(CONFIG["out_dir"], f"tsne_{tag}.png"),
            n_samples=CONFIG["tsne_samples"]
        )

    # ── FID (sample 2048 fake images) ────────────────────────────
    model.eval()
    with torch.no_grad():
        z_fid    = torch.randn(2048, latent_dim, device=DEVICE)
        fake_fid = model.decode(z_fid).view(-1, 1, 28, 28)
    fid_score = compute_fid(test_loader, fake_fid, DEVICE)
    print(f"  FID = {fid_score:.2f}")

    # Save checkpoint
    ckpt_path = os.path.join(CONFIG["out_dir"], f"vae_{tag}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"  [✓] Checkpoint → {ckpt_path}")

    torch.cuda.empty_cache()
    return {
        "Dataset"       : dataset_name.upper(),
        "Latent_dim"    : latent_dim,
        "Recon_MSE"     : round(recon_mse, 6),
        "FID"           : round(fid_score, 2),
        "Train_time_min": round(train_time_min, 2),
        "Epochs"        : epochs,
    }


# ════════════════════════════════════════════════════════════════
# TASK 1 & 2 & 3 & 4 & 6 — Primary run (z=32, FashionMNIST)
# ════════════════════════════════════════════════════════════════
results_fmnist = run_vae_experiment(
    latent_dim   = CONFIG["primary_dim"],
    dataset_name = "fmnist",
    epochs       = CONFIG["epochs"],
    do_visualise = True
)

# ── Primary run on EMNIST ────────────────────────────────────────
results_emnist = run_vae_experiment(
    latent_dim   = CONFIG["primary_dim"],
    dataset_name = "emnist",
    epochs       = CONFIG["epochs"],
    do_visualise = True
)

# ════════════════════════════════════════════════════════════════
# TASK 5 — Latent Dim Study: z ∈ {8, 32, 128} on FashionMNIST
# ════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  TASK 5: Latent Dimension Ablation Study")
print("="*60)

dim_study_results = []
for z_dim in CONFIG["latent_dims"]:
    res = run_vae_experiment(
        latent_dim   = z_dim,
        dataset_name = "fmnist",
        epochs       = CONFIG["epochs"],
        do_visualise = (z_dim == CONFIG["primary_dim"])  # skip re-vis for z=32
    )
    dim_study_results.append(res)

# ── Summary Table ────────────────────────────────────────────────
df_dims = pd.DataFrame(dim_study_results)
print("\n── Latent Dim Study Results ──")
print(df_dims.to_string(index=False))
df_dims.to_csv(os.path.join(CONFIG["out_dir"], "latent_dim_study.csv"), index=False)

# ── Bar chart: FID vs latent_dim ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fid_vals = [r["FID"]       for r in dim_study_results]
mse_vals = [r["Recon_MSE"] for r in dim_study_results]
z_labels = [f"z={r['Latent_dim']}" for r in dim_study_results]

bars0 = axes[0].bar(z_labels, fid_vals, color=['#3498DB', '#E74C3C', '#2ECC71'], edgecolor='black')
axes[0].set_title('FID vs Latent Dim (FashionMNIST)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('FID ↓'); axes[0].set_xlabel('Latent Dimension')
axes[0].bar_label(bars0, fmt='%.1f', padding=3)
axes[0].grid(axis='y', alpha=0.3)

bars1 = axes[1].bar(z_labels, mse_vals, color=['#3498DB', '#E74C3C', '#2ECC71'], edgecolor='black')
axes[1].set_title('Recon MSE vs Latent Dim', fontsize=13, fontweight='bold')
axes[1].set_ylabel('MSE ↓'); axes[1].set_xlabel('Latent Dimension')
axes[1].bar_label(bars1, fmt='%.5f', padding=3)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('VAE — Latent Dimension Ablation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "latent_dim_ablation.png"), dpi=150, bbox_inches='tight')
plt.close()
print("[✓] Ablation chart saved.")

torch.cuda.empty_cache()
print("\n✅ Problem 1 — VAE Complete!")
print(f"   All outputs saved to: {CONFIG['out_dir']}")